# Tổng hợp chính thức một mô hình trên 3 seed
Notebook không train lại mô hình. Hãy Add Input đủ output của seed 46, 48 và 50 cho cả hai dataset.

Kết quả của hai dataset được tổng hợp riêng, không lấy trung bình gộp.

## 1. Thu thập và kiểm tra artefact
Kiểm tra đúng backbone, đủ run, đủ sáu phương pháp và matched-budget trước khi tổng hợp.

In [ ]:
BACKBONE = "resnet50"
SEEDS = [46, 48, 50]
EXPECTED_RUN_IDS = []
GIT_COMMIT = "REPLACE_GIT_COMMIT"

## 2. Mean và sample standard deviation trên 3 seed

In [ ]:
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient
repo = Path('/kaggle/working/neuro_symbolic_mlops_l2_app')
if not repo.exists():
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-b', 'rollback', '--single-branch', f'https://{token}@github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', GIT_COMMIT], check=True)

In [ ]:
cmd = [
    'python', '-m', 'pipelines.bundle_model_outputs',
    '--input-root', '/kaggle/input',
    '--output-dir', f'/kaggle/working/bundle_{BACKBONE}',
    '--backbone', BACKBONE,
    '--seeds', *map(str, SEEDS),
    '--expected-run-ids', *EXPECTED_RUN_IDS,
]
subprocess.run(cmd, cwd=repo, check=True)

In [ ]:
import pandas as pd
bundle_dir = Path(f'/kaggle/working/bundle_{BACKBONE}')
print('Tổng hợp mean/std trên 3 seed:', SEEDS)
summary = pd.read_csv(bundle_dir / 'model_three_seed_summary.csv')
official = pd.read_csv(bundle_dir / 'official_experiment_comparison.csv')
display(summary)
display(official)

## 3. Paired delta theo seed

In [ ]:
display(pd.read_csv(bundle_dir / 'paired_delta_vs_cnn.csv'))
display(pd.read_csv(bundle_dir / 'bayesian_vs_core_methods.csv'))

## 4. Fairness, chất lượng luật và ranking analysis

In [ ]:
display(pd.read_csv(bundle_dir / 'matched_budget_audit.csv'))
display(pd.read_csv(bundle_dir / 'rule_set_quality_mean_std.csv'))
display(pd.read_csv(bundle_dir / 'rule_ranking_metrics_mean_std.csv'))

## 5. Danh sách file xuất

In [ ]:
for path in sorted(bundle_dir.glob('*')):
    if path.is_file():
        print(path.name)